In [ ]:
import sys
from pathlib import Path

def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / ".git").exists():
            return p
    raise RuntimeError("Could not locate project root (no .git found above cwd)")

PROJECT_ROOT = _find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments" / "mlp" / "experience_replay"


# MLP Experience Replay with Confidently Correct Memory
Train an MLPClassifier with year-wise incremental scaling and experience replay prioritizing confidently correct samples.

In [ ]:
import copy
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from tqdm.notebook import tqdm

from src.mlp_replay.data import (
    load_dataset_and_splits,
    prepare_raw_features_for_year,
    prepare_features_for_year,
    precompute_yearly_raw_cache,
)
from src.mlp_replay.model import (
    build_mlp_model,
    capture_model_state,
    restore_model_state,
    compute_year_positive_rate,
    compute_binary_class_weights,
    compute_optimal_f1_threshold,
)
from src.mlp_replay.replay_strategies import (
    weighted_choice_without_replacement,
    sample_confidently_correct_replay_indices,
    sample_random_replay_indices,
)
from src.mlp_replay.training_loop import (
    resume_or_init_ratio_state,
    load_year_batch_or_none,
    finalize_ratio_and_save,
    write_combined_history_csv,
    merge_all_ratio_histories,
)
from src.mlp_replay.checkpointing import (
    create_empty_training_history,
    load_all_training_histories,
    save_all_training_histories,
    load_completion_status,
    save_completion_status,
    append_training_log,
    format_ratio_key,
    get_cached_raw_year,
    per_ratio_path,
)

warnings.filterwarnings('ignore')

## 1. Load Data and Splits

In [ ]:
ds_path = PROJECT_ROOT / 'training_data_with_features_plus_monthly_indices.zarr'
split_path = PROJECT_ROOT / 'data_split.npz'
ds, train_pixel_indices, val_pixel_indices, test_pixel_indices = load_dataset_and_splits(ds_path, split_path)


## 2. Feature Engineering

`prepare_raw_features_for_year` and `prepare_features_for_year` now live in `src/mlp_replay/data.py` and are imported above.


## 3. Initialize Class Weights and MLP

In [ ]:
print('Precomputing raw yearly features for train/validation splits...')
n_years = len(ds.year)


train_feature_cache = precompute_yearly_raw_cache(ds, train_pixel_indices, n_years, 'train', ds_path=ds_path)
val_feature_cache = precompute_yearly_raw_cache(ds, val_pixel_indices, n_years, 'validation', ds_path=ds_path)

classes = np.array([0, 1])

model = MLPClassifier(
    hidden_layer_sizes=(64,),
    activation='relu',
    alpha=0.0001,
    random_state=42,
    solver='adam',
    learning_rate='adaptive',
    max_iter=1,
    learning_rate_init=0.001,
    warm_start=False,
    verbose=False,
)
print('MLP initialized')

## 4. Replay-Enabled Online Training

### Replay Sampling Strategy (Confidently Correct + Random)
For each training year, replay samples are split into two parts:
- **20% confidently correct memory replay**
- **80% uniform random replay**

Confidently correct memory is sampled as follows:
1. Identify all previous years with available replay samples.
2. Split the confidently correct replay target as evenly as possible across these previous years.
3. For each previous year, run current-model probability predictions on all that year's replay candidates.
4. Compute the **optimal F1 threshold** for that year from probabilities and labels (threshold grid search).
5. Determine correct classifications where predicted class matches the true class: $y_{pred} == y_{true}$, where $y_{pred} = 1$ if $p_i \ge t^*$ else $0$, and $t^*$ is the optimal F1 threshold.
6. For correctly classified samples, compute weights based on the probability of the true class (confidence):

$$w_i = 1.0 - |y_i - p_i|$$

For incorrectly classified samples, set weight to 0.0.

7. Sample from that year's candidates using these weights.

The random part is sampled uniformly from remaining replay candidates to avoid overlap with confidently correct-selected samples.

In [ ]:
import json
from datetime import datetime

REPLAY_RATIOS = [0.2, 0.3, 0.4, 0.5]

import os
_replay_ratio_override = os.environ.get('MLP_REPLAY_RATIO_OVERRIDE')
if _replay_ratio_override:
    REPLAY_RATIOS = [float(_replay_ratio_override)]
REPLAY_ENABLED = True
REPLAY_RANDOM_STATE = 42
CONFIDENTLY_CORRECT_REPLAY_FRACTION = 0.2
THRESHOLD_GRID = np.linspace(0.0, 1.0, 201)
EPSILON = 1e-12

WEIGHT_POLICY_NAME = 'post_sampling_replay_class_weights_v1'
REPLAY_WEIGHT_FALLBACK = 'smoothed_single_class'
REPLAY_POSITIVE_LABEL = 1

CHUNK_SIZE = 50000
MAX_EPOCHS = 15
PATIENCE = 3
MIN_DELTA = 0.0005

if REPLAY_WEIGHT_FALLBACK not in {'smoothed_single_class'}:
    raise ValueError(f'Unsupported REPLAY_WEIGHT_FALLBACK: {REPLAY_WEIGHT_FALLBACK}')

CHECKPOINT_DIR = EXPERIMENTS_DIR / 'training_checkpoints_mlp_experience_replay_confidently_correct_memory'
ALL_HISTORIES_FILE = CHECKPOINT_DIR / 'mlp_replay_all_training_histories_confidently_correct_memory.pkl'
COMPLETION_STATUS_FILE = CHECKPOINT_DIR / 'mlp_replay_completion_status_confidently_correct_memory.json'
TRAINING_LOG_FILE = CHECKPOINT_DIR / 'mlp_replay_training_log_confidently_correct_memory.txt'
COMBINED_HISTORY_FILENAME = 'mlp_classifier_history_prevyears_monthly_features_incremental_scaler_experience_replay_confidently_correct_memory_all_ratios.csv'

CHECKPOINT_DIR.mkdir(exist_ok=True)

if 'train_feature_cache' not in globals() or 'val_feature_cache' not in globals() or not train_feature_cache or not val_feature_cache:
    raise ValueError('Raw feature caches not found or empty. Run Cell 8 first to precompute yearly features.')


n_years = len(ds.year)
year_values = ds.year.values
year_value_to_idx = {int(y): idx for idx, y in enumerate(year_values)}
all_target_year_values = [int(year_values[idx]) for idx in range(1, n_years)]

print(f'Checkpoint directory: {CHECKPOINT_DIR.resolve()}')
print(f'Replay ratios: {REPLAY_RATIOS}')
print(f'Weight policy: {WEIGHT_POLICY_NAME}, replay_weight_fallback={REPLAY_WEIGHT_FALLBACK}')
append_training_log(TRAINING_LOG_FILE, f'Started training run for ratios: {REPLAY_RATIOS}')

for ratio_idx, replay_ratio in enumerate(REPLAY_RATIOS):
    ratio_key = format_ratio_key(replay_ratio)
    ratio_histories_file = per_ratio_path(ALL_HISTORIES_FILE, ratio_key)
    ratio_completion_file = per_ratio_path(COMPLETION_STATUS_FILE, ratio_key)
    ratio_log_file = per_ratio_path(TRAINING_LOG_FILE, ratio_key)
    all_training_histories = load_all_training_histories(ratio_histories_file)
    completion_status = load_completion_status(ratio_completion_file, extra_default_keys=['weight_policy_by_ratio'])
    completion_status.setdefault('weight_policy_by_ratio', {})
    output_suffix = f'incremental_scaler_experience_replay_confidently_correct_memory_{ratio_key}'
    models_dir_name = f'models_mlp_prevyears_monthly_features_{output_suffix}'
    scaler_file_template = f'scaler_year_{{year}}_mlp_prevyears_monthly_features_{output_suffix}.pkl'
    model_file_template = f'model_year_{{year}}_mlp_prevyears_monthly_features_{output_suffix}.pkl'
    final_scaler_filename = f'scaler_final_mlp_prevyears_monthly_features_{output_suffix}.pkl'
    final_model_filename = f'mlp_classifier_model_prevyears_monthly_features_{output_suffix}.pkl'
    history_filename = f'mlp_classifier_history_prevyears_monthly_features_{output_suffix}.csv'

    models_dir = EXPERIMENTS_DIR / models_dir_name
    models_dir.mkdir(exist_ok=True)

    expected_policy_config = {
        'weight_policy': WEIGHT_POLICY_NAME,
        'replay_weight_fallback': REPLAY_WEIGHT_FALLBACK,
        'weight_source': 'current_year_and_post_sampling_replay',
    }
    completion_status['weight_policy_by_ratio'][ratio_key] = expected_policy_config

    if ratio_key in completion_status.get('completed_ratios', []):
        print(f'[{ratio_key}] already completed. Skipping ratio.')
        append_training_log(ratio_log_file, f'[{ratio_key}] skipped (already completed).')
        continue
    replay_rng = np.random.default_rng(REPLAY_RANDOM_STATE + ratio_idx)

    model, incremental_scaler, start_year_idx, completed_years, training_history = resume_or_init_ratio_state(
        ratio_key, models_dir, model_file_template, scaler_file_template, year_value_to_idx,
        all_training_histories, completion_status, ratio_log_file,
        extra_history_keys=['confidently_correct_target_size', 'confidently_correct_used_size', 'random_target_size', 'random_used_size', 'threshold_mean', 'weight_policy', 'year_positive_rate', 'replay_actual_positive_rate', 'replay_pool_positive_rate', 'current_class_weight_0', 'current_class_weight_1', 'replay_class_weight_0', 'replay_class_weight_1', 'replay_weight_fallback'],
    )

    print(f'[{ratio_key}] Model output directory: {models_dir.resolve()}')
    print(f'[{ratio_key}] Years to train: 1..{n_years - 1} (year 0 skipped)')

    for year_idx in tqdm(range(start_year_idx, n_years), desc=f'{ratio_key} by year'):
        year_val = int(year_values[year_idx])

        if year_val in completed_years:
            continue

        year_batch = load_year_batch_or_none(
            train_feature_cache, val_feature_cache, year_idx, year_val, ratio_key,
            completed_years, completion_status, ratio_completion_file, ratio_log_file,
        )
        if year_batch is None:
            continue
        X_train_raw, y_train_batch, X_val_raw, y_val_batch = year_batch

        year_positive_rate = compute_year_positive_rate(y_train_batch, positive_label=REPLAY_POSITIVE_LABEL)
        current_class_weight_dict, current_weight_fallback = compute_binary_class_weights(
            y_train_batch,
            classes=classes,
            fallback_mode=REPLAY_WEIGHT_FALLBACK,
        )

        incremental_scaler.partial_fit(X_train_raw)
        X_train_batch = incremental_scaler.transform(X_train_raw)
        X_val_batch = incremental_scaler.transform(X_val_raw)

        scaler_checkpoint = copy.deepcopy(incremental_scaler)
        year_scaler_path = models_dir / scaler_file_template.format(year=year_val)
        with open(year_scaler_path, 'wb') as f:
            pickle.dump(scaler_checkpoint, f)

        n_samples = len(X_train_batch)
        replay_target_size = int(n_samples * replay_ratio) if REPLAY_ENABLED else 0

        replay_X_parts = []
        replay_y_parts = []
        replay_year_spans = []
        replay_cursor = 0

        if REPLAY_ENABLED and year_idx > 1:
            for past_year_idx in range(1, year_idx):
                X_past_raw, y_past = get_cached_raw_year(train_feature_cache, past_year_idx)
                if len(X_past_raw) > 0:
                    X_past = incremental_scaler.transform(X_past_raw)
                    replay_X_parts.append(X_past)
                    replay_y_parts.append(y_past)
                    span_len = len(y_past)
                    replay_year_spans.append((past_year_idx, replay_cursor, replay_cursor + span_len))
                    replay_cursor += span_len

        if replay_X_parts:
            X_replay_pool = np.vstack(replay_X_parts)
            y_replay_pool = np.concatenate(replay_y_parts)
        else:
            X_replay_pool = np.empty((0, X_train_batch.shape[1]), dtype=X_train_batch.dtype)
            y_replay_pool = np.empty((0,), dtype=y_train_batch.dtype)

        replay_pool_size = len(y_replay_pool)
        replay_used_size = min(replay_target_size, replay_pool_size) if REPLAY_ENABLED else 0
        replay_actual_positive_rate = np.nan
        replay_pool_positive_rate = (
            float(np.mean(y_replay_pool == REPLAY_POSITIVE_LABEL))
            if replay_pool_size > 0
            else np.nan
        )

        if hasattr(model, 'n_features_in_') and int(model.n_features_in_) != int(X_train_batch.shape[1]):
            raise ValueError(
                f'Feature count mismatch at year {year_val}: model expects {model.n_features_in_}, got {X_train_batch.shape[1]}'
            )

        best_val_pr_auc = -np.inf
        patience_counter = 0
        best_model_state = None
        confident_used_size = 0
        random_used_size = 0
        confident_target_size = 0
        random_target_size = 0
        threshold_mean = np.nan
        current_effective_mean_weight = np.nan
        replay_effective_mean_weight = np.nan
        replay_class_weight_dict = {0: np.nan, 1: np.nan}
        replay_weight_fallback = 'no_replay'

        for epoch in range(MAX_EPOCHS):
            if replay_used_size > 0:
                confident_target_size = int(np.floor(replay_used_size * CONFIDENTLY_CORRECT_REPLAY_FRACTION))
                random_target_size = int(replay_used_size - confident_target_size)

                confident_indices, threshold_values = sample_confidently_correct_replay_indices(
                    model=model,
                    X_replay_pool=X_replay_pool,
                    y_replay_pool=y_replay_pool,
                    replay_year_spans=replay_year_spans,
                    confident_target_size=confident_target_size,
                    rng=replay_rng,
                    threshold_grid=THRESHOLD_GRID,
                )
                random_indices = sample_random_replay_indices(
                    replay_pool_size=replay_pool_size,
                    random_target_size=random_target_size,
                    excluded_indices=confident_indices,
                    rng=replay_rng,
                )

                replay_indices = np.concatenate([confident_indices, random_indices])
                if len(replay_indices) > 0:
                    replay_indices = replay_indices.astype(np.int64, copy=False)
                    replay_indices = replay_indices[replay_rng.permutation(len(replay_indices))]

                confident_used_size = int(len(confident_indices))
                random_used_size = int(len(random_indices))
                if threshold_values:
                    threshold_mean = float(np.mean(threshold_values))
                else:
                    threshold_mean = np.nan

                if len(replay_indices) > 0:
                    X_replay_sampled = X_replay_pool[replay_indices]
                    y_replay_sampled = y_replay_pool[replay_indices]
                    replay_actual_positive_rate = float(np.mean(y_replay_sampled == REPLAY_POSITIVE_LABEL))
                    replay_class_weight_dict, replay_weight_fallback = compute_binary_class_weights(
                        y_replay_sampled,
                        classes=classes,
                        fallback_mode=REPLAY_WEIGHT_FALLBACK,
                    )

                    X_combined = np.concatenate([X_train_batch, X_replay_sampled], axis=0)
                    y_combined = np.concatenate([y_train_batch, y_replay_sampled], axis=0)
                    origin_is_replay_combined = np.concatenate(
                        [
                            np.zeros(len(y_train_batch), dtype=bool),
                            np.ones(len(y_replay_sampled), dtype=bool),
                        ]
                    )
                    replay_used_size = int(len(y_replay_sampled))
                else:
                    X_combined = X_train_batch
                    y_combined = y_train_batch
                    origin_is_replay_combined = np.zeros(len(y_train_batch), dtype=bool)
                    replay_used_size = 0
                    replay_actual_positive_rate = np.nan
                    replay_class_weight_dict = {0: np.nan, 1: np.nan}
                    replay_weight_fallback = 'no_replay'
            else:
                X_combined = X_train_batch
                y_combined = y_train_batch
                origin_is_replay_combined = np.zeros(len(y_train_batch), dtype=bool)
                replay_used_size = 0
                replay_actual_positive_rate = np.nan
                replay_class_weight_dict = {0: np.nan, 1: np.nan}
                replay_weight_fallback = 'no_replay'

            combined_n_samples = len(X_combined)
            shuffle_idx = replay_rng.permutation(combined_n_samples)
            X_train_shuffled = X_combined[shuffle_idx]
            y_train_shuffled = y_combined[shuffle_idx]
            origin_is_replay_shuffled = origin_is_replay_combined[shuffle_idx]
            n_chunks = max(1, int(np.ceil(combined_n_samples / CHUNK_SIZE)))

            for chunk_idx in range(n_chunks):
                start_idx = chunk_idx * CHUNK_SIZE
                end_idx = min(start_idx + CHUNK_SIZE, combined_n_samples)
                X_chunk = X_train_shuffled[start_idx:end_idx]
                y_chunk = y_train_shuffled[start_idx:end_idx]
                origin_is_replay_chunk = origin_is_replay_shuffled[start_idx:end_idx]

                sample_weights_chunk = np.empty(len(y_chunk), dtype=np.float64)
                current_chunk_mask = ~origin_is_replay_chunk
                replay_chunk_mask = origin_is_replay_chunk
                if np.any(current_chunk_mask):
                    sample_weights_chunk[current_chunk_mask] = np.array(
                        [current_class_weight_dict[int(label)] for label in y_chunk[current_chunk_mask]],
                        dtype=np.float64,
                    )
                if np.any(replay_chunk_mask):
                    sample_weights_chunk[replay_chunk_mask] = np.array(
                        [replay_class_weight_dict[int(label)] for label in y_chunk[replay_chunk_mask]],
                        dtype=np.float64,
                    )

                if len(sample_weights_chunk) != len(y_chunk):
                    raise ValueError(
                        f'Sample weight length mismatch: weights={len(sample_weights_chunk)} vs labels={len(y_chunk)}'
                    )
                if np.any(~np.isfinite(sample_weights_chunk)):
                    raise ValueError('Encountered non-finite sample weights in training chunk.')
                if np.any(sample_weights_chunk <= 0):
                    raise ValueError('Encountered non-positive sample weights in training chunk.')

                model.partial_fit(X_chunk, y_chunk, classes=classes, sample_weight=sample_weights_chunk)

            current_mask = ~origin_is_replay_shuffled
            replay_mask = origin_is_replay_shuffled
            if np.any(current_mask):
                current_effective_mean_weight = float(
                    np.mean(
                        np.array([current_class_weight_dict[int(label)] for label in y_train_shuffled[current_mask]], dtype=np.float64)
                    )
                )
            if np.any(replay_mask):
                replay_effective_mean_weight = float(
                    np.mean(
                        np.array([replay_class_weight_dict[int(label)] for label in y_train_shuffled[replay_mask]], dtype=np.float64)
                    )
                )

            y_val_pred = model.predict(X_val_batch)
            y_val_proba = model.predict_proba(X_val_batch)[:, 1]
            val_pr_auc = average_precision_score(y_val_batch, y_val_proba) if len(np.unique(y_val_batch)) > 1 else np.nan

            if val_pr_auc > best_val_pr_auc + MIN_DELTA:
                best_val_pr_auc = val_pr_auc
                patience_counter = 0
                best_model_state = capture_model_state(model)
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    break

        if best_model_state is not None:
            restore_model_state(model, best_model_state)

        y_train_pred = model.predict(X_train_batch)
        y_val_pred = model.predict(X_val_batch)
        y_val_proba = model.predict_proba(X_val_batch)[:, 1]

        train_acc = accuracy_score(y_train_batch, y_train_pred)
        train_prec = precision_score(y_train_batch, y_train_pred, zero_division=0)
        train_rec = recall_score(y_train_batch, y_train_pred, zero_division=0)
        train_f1 = f1_score(y_train_batch, y_train_pred, zero_division=0)

        val_acc = accuracy_score(y_val_batch, y_val_pred)
        val_prec = precision_score(y_val_batch, y_val_pred, zero_division=0)
        val_rec = recall_score(y_val_batch, y_val_pred, zero_division=0)
        val_f1 = f1_score(y_val_batch, y_val_pred, zero_division=0)
        if len(np.unique(y_val_batch)) > 1:
            val_roc_auc = roc_auc_score(y_val_batch, y_val_proba)
            val_pr_auc = average_precision_score(y_val_batch, y_val_proba)
        else:
            val_roc_auc = np.nan
            val_pr_auc = np.nan

        training_history['year'].append(year_val)
        training_history['train_accuracy'].append(train_acc)
        training_history['train_precision'].append(train_prec)
        training_history['train_recall'].append(train_rec)
        training_history['train_f1'].append(train_f1)
        training_history['val_accuracy'].append(val_acc)
        training_history['val_precision'].append(val_prec)
        training_history['val_recall'].append(val_rec)
        training_history['val_f1'].append(val_f1)
        training_history['val_roc_auc'].append(val_roc_auc)
        training_history['val_pr_auc'].append(val_pr_auc)
        training_history['replay_pool_size'].append(int(replay_pool_size))
        training_history['replay_target_size'].append(int(replay_target_size))
        training_history['replay_used_size'].append(int(replay_used_size))
        training_history['confidently_correct_target_size'].append(int(confident_target_size))
        training_history['confidently_correct_used_size'].append(int(confident_used_size))
        training_history['random_target_size'].append(int(random_target_size))
        training_history['random_used_size'].append(int(random_used_size))
        training_history['threshold_mean'].append(float(threshold_mean) if not np.isnan(threshold_mean) else np.nan)
        training_history['weight_policy'].append(WEIGHT_POLICY_NAME)
        training_history['year_positive_rate'].append(float(year_positive_rate))
        training_history['replay_actual_positive_rate'].append(float(replay_actual_positive_rate) if replay_used_size > 0 else np.nan)
        training_history['replay_pool_positive_rate'].append(float(replay_pool_positive_rate) if replay_pool_size > 0 else np.nan)
        training_history['current_class_weight_0'].append(float(current_class_weight_dict[0]))
        training_history['current_class_weight_1'].append(float(current_class_weight_dict[1]))
        training_history['replay_class_weight_0'].append(float(replay_class_weight_dict[0]) if replay_used_size > 0 else np.nan)
        training_history['replay_class_weight_1'].append(float(replay_class_weight_dict[1]) if replay_used_size > 0 else np.nan)
        training_history['replay_weight_fallback'].append(replay_weight_fallback)

        year_model_path = models_dir / model_file_template.format(year=year_val)
        with open(year_model_path, 'wb') as f:
            pickle.dump(model, f)

        completed_years.add(year_val)
        completion_status['completed_years'][ratio_key] = sorted(list(completed_years))
        all_training_histories[ratio_key] = training_history.copy()
        save_all_training_histories(ratio_histories_file, all_training_histories)
        save_completion_status(ratio_completion_file, completion_status)

        print(
            f'[{ratio_key}] Year {year_val}: Train F1={train_f1:.3f}, Val F1={val_f1:.3f}, Val PR-AUC={val_pr_auc:.3f}, '
            f'replay_used={replay_used_size:,}/{replay_pool_size:,}, replay_pos_rate={replay_actual_positive_rate:.2%}, '
            f'year_pos_rate={year_positive_rate:.2%}, replay_w0={replay_class_weight_dict[0]:.3f}, replay_w1={replay_class_weight_dict[1]:.3f}'
        )
        append_training_log(
            ratio_log_file,
            f'[{ratio_key}] completed year {year_val} with replay_used={replay_used_size}/{replay_pool_size}, '
            f'confidently_correct={confident_used_size}, random={random_used_size}, replay_pos_rate={replay_actual_positive_rate:.4f}, '
            f'year_pos_rate={year_positive_rate:.4f}, replay_pool_pos_rate={replay_pool_positive_rate:.4f}, '
            f'current_w0={current_class_weight_dict[0]:.4f}, current_w1={current_class_weight_dict[1]:.4f}, '
            f'replay_w0={replay_class_weight_dict[0]:.4f}, replay_w1={replay_class_weight_dict[1]:.4f}, '
            f'replay_weight_fallback={replay_weight_fallback}, '
            f'current_effective_mean_weight={current_effective_mean_weight:.4f}, replay_effective_mean_weight={replay_effective_mean_weight:.4f}.',
        )

    finalize_ratio_and_save(
        incremental_scaler, model, models_dir, final_scaler_filename, final_model_filename,
        history_filename, training_history, ratio_key, all_target_year_values, completed_years,
        completion_status, ratio_completion_file, ratio_log_file, all_training_histories, ratio_histories_file,
    )

merged_histories = merge_all_ratio_histories(CHECKPOINT_DIR, ALL_HISTORIES_FILE.stem)
write_combined_history_csv(merged_histories, EXPERIMENTS_DIR / COMBINED_HISTORY_FILENAME)

append_training_log(TRAINING_LOG_FILE, 'Training run completed.')

## 5. Save Training History

In [6]:
import json

CHECKPOINT_DIR = EXPERIMENTS_DIR / 'training_checkpoints_mlp_experience_replay_confidently_correct_memory'
ALL_HISTORIES_FILE = CHECKPOINT_DIR / 'mlp_replay_all_training_histories_confidently_correct_memory.pkl'
COMPLETION_STATUS_FILE = CHECKPOINT_DIR / 'mlp_replay_completion_status_confidently_correct_memory.json'
COMBINED_HISTORY_FILENAME = 'mlp_classifier_history_prevyears_monthly_features_incremental_scaler_experience_replay_confidently_correct_memory_all_ratios.csv'

all_training_histories = merge_all_ratio_histories(CHECKPOINT_DIR, ALL_HISTORIES_FILE.stem)

summary_rows = []
for ratio_key in sorted(all_training_histories.keys()):
    history_dict = all_training_histories[ratio_key]
    n_rows = len(history_dict.get('year', []))
    last_year = history_dict['year'][-1] if n_rows > 0 else np.nan
    last_val_f1 = history_dict['val_f1'][-1] if n_rows > 0 else np.nan
    last_val_pr_auc = history_dict['val_pr_auc'][-1] if n_rows > 0 else np.nan
    ratio_completion_status = load_completion_status(per_ratio_path(COMPLETION_STATUS_FILE, ratio_key))
    is_completed = ratio_key in ratio_completion_status.get('completed_ratios', [])
    summary_rows.append(
        {
            'ratio_key': ratio_key,
            'rows': n_rows,
            'last_year': last_year,
            'last_val_f1': last_val_f1,
            'last_val_pr_auc': last_val_pr_auc,
            'completed': is_completed,
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values('ratio_key').reset_index(drop=True)
print('Per-ratio training summary:')
display(summary_df)

combined_history_path = EXPERIMENTS_DIR / COMBINED_HISTORY_FILENAME
if combined_history_path.exists():
    combined_df = pd.read_csv(combined_history_path)
    print(f'Combined history file: {combined_history_path}')
    print(f'Rows: {len(combined_df):,}')
    display(combined_df.tail())
else:
    print(f'Combined history file not found: {combined_history_path}')

Per-ratio training summary:


,ratio_key,rows,last_year,last_val_f1,last_val_pr_auc,completed
0,RR_0.2,0,NaN,NaN,NaN,True
1,RR_0.3,0,NaN,NaN,NaN,True
2,RR_0.4,0,NaN,NaN,NaN,True
3,RR_0.5,0,NaN,NaN,NaN,True


Combined history file not found: mlp_classifier_history_prevyears_monthly_features_incremental_scaler_experience_replay_confidently_correct_memory_all_ratios.csv
